In [1]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import HuberRegressor

/Users/ywo/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# author's datashare
df = pd.read_csv('/Users/ywo/Downloads/Industry Project/datashare (1)/datashare.csv')
df['DATE']=pd.to_datetime(df['DATE'], format='%Y%m%d')

In [3]:
db = wrds.Connection()

Enter your WRDS username [ywo]:ywhan
Enter your password:········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
permnos = df['permno'].unique().tolist()
dates = df['DATE'].unique().tolist()
start_date = min(dates)
end_date = max(dates)


query = f"""
SELECT 
    b.permno, 
    b.date, 
    b.ret, 
    c.rf,
    (b.ret - c.rf) AS exret
FROM crsp.msf b
LEFT JOIN ff.factors_monthly c ON 
    EXTRACT(YEAR FROM b.date) = EXTRACT(YEAR FROM c.date)
    AND EXTRACT(MONTH FROM b.date) = EXTRACT(MONTH FROM c.date)
WHERE b.date >= '{start_date}' AND b.date <= '{end_date}'
  AND b.ret IS NOT NULL  
"""

df_msf_all = db.raw_sql(query)
df['DATE'] = pd.to_datetime(df['DATE'])
df_msf_all['date'] = pd.to_datetime(df_msf_all['date'])

df_exret = pd.merge(
    df, 
    df_msf_all,
    left_on=['permno', 'DATE'],
    right_on=['permno', 'date'],
    how='inner'
)

In [5]:
df_exret.drop(columns=['date'])
df_exret=df_exret[df_exret['DATE']>='1957-03-01']

In [6]:
# 8 macro variables from websites
df_macro=pd.read_excel('/Users/ywo/Downloads/Industry Project/datashare (1)/Data2024.xlsx', sheet_name='Monthly')
macro=['tbl','d/p','e/p','b/m','tms','dfy','ntis','svar']
df_macro[macro]=df_macro[macro].shift(1)


In [7]:
df_macro['date'] = pd.to_datetime(df_macro['yyyymm'], format='%Y%m')
df_macro['year_month'] = df_macro['date'].dt.to_period('M')


In [8]:
df_macro=df_macro[['yyyymm','tbl','d/p','e/p','b/m','tms','dfy','ntis','svar','year_month']]
# df_macro

In [9]:
df_exret['year_month'] = df_exret['DATE'].dt.to_period('M')
df_exret = df_exret[df_exret['ret'].notna()]
df_exret = df_exret[df_exret['mvel1'].notna()]
df_exret = df_exret.reset_index(drop=True)
# df_exret

In [10]:
features=list(df_exret.columns)[2:96]
# features

In [11]:
def norm_rank(data):
    median_val = data.median()
    filled_data = data.fillna(median_val).fillna(0)
    ranks = filled_data.rank(method='average')
    n = len(filled_data)
    mapped = (ranks / (n + 1)) * 2 - 1
    return mapped
df_exret[features]=df_exret.groupby('DATE')[features].apply(norm_rank).reset_index(drop=True)
# df_exret

In [12]:
# final data
df_merged=pd.merge(df_exret,df_macro,on='year_month',how='left')
df_merged.drop(columns=['yyyymm','year_month'])

,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,rf,exret,tbl,d/p,e/p,b/m,tms,dfy,ntis,svar
0,10006,1957-03-29,0.313800,0.219282,0.219282,0.803403,0.724008,-0.378072,0.000000,0.470699,...,0.0023,0.016105,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
1,10014,1957-03-29,-0.943289,-0.905482,-0.907372,0.381853,-0.708885,0.969754,0.000000,-0.832703,...,0.0023,-0.0023,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
2,10022,1957-03-29,-0.720227,-0.141777,-0.141777,0.608696,-0.644612,0.071834,0.000000,-0.273157,...,0.0023,-0.006146,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
3,10030,1957-03-29,0.056711,-0.224953,-0.224953,0.156900,0.379962,-0.604915,0.000000,0.416824,...,0.0023,0.075607,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
4,10057,1957-03-29,-0.122873,0.362949,0.362949,-0.200378,-0.196597,-0.321361,0.000000,-0.158790,...,0.0023,-0.020031,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4091701,93423,2021-12-31,0.522734,0.855778,0.854909,-0.719954,0.712134,0.348103,0.823052,-0.481610,...,0.0001,0.164242,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327
4091702,93426,2021-12-31,-0.109470,0.385462,0.382566,0.561251,-0.306111,-0.598031,0.305097,0.542427,...,0.0001,0.08117,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327
4091703,93427,2021-12-31,0.587895,-0.406313,-0.410078,0.520996,0.385752,-0.305242,0.305097,0.909933,...,0.0001,0.071445,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327
4091704,93434,2021-12-31,-0.624095,-0.795251,-0.799884,-0.613380,-0.358529,0.609036,0.220533,-0.931364,...,0.0001,-0.065169,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327


In [13]:
df_merged['sic2'] = df_merged.groupby('permno')['sic2'].ffill().bfill()

,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,year_month,yyyymm,tbl,d/p,e/p,b/m,tms,dfy,ntis,svar
0,10006,1957-03-29,0.313800,0.219282,0.219282,0.803403,0.724008,-0.378072,0.000000,0.470699,...,1957-03,195703,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
1,10014,1957-03-29,-0.943289,-0.905482,-0.907372,0.381853,-0.708885,0.969754,0.000000,-0.832703,...,1957-03,195703,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
2,10022,1957-03-29,-0.720227,-0.141777,-0.141777,0.608696,-0.644612,0.071834,0.000000,-0.273157,...,1957-03,195703,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
3,10030,1957-03-29,0.056711,-0.224953,-0.224953,0.156900,0.379962,-0.604915,0.000000,0.416824,...,1957-03,195703,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
4,10057,1957-03-29,-0.122873,0.362949,0.362949,-0.200378,-0.196597,-0.321361,0.000000,-0.158790,...,1957-03,195703,0.0310,0.040068,0.078672,0.584994,0.0018,0.0080,0.030174,0.001056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4091701,93423,2021-12-31,0.522734,0.855778,0.854909,-0.719954,0.712134,0.348103,0.823052,-0.481610,...,2021-12,202112,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327
4091702,93426,2021-12-31,-0.109470,0.385462,0.382566,0.561251,-0.306111,-0.598031,0.305097,0.542427,...,2021-12,202112,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327
4091703,93427,2021-12-31,0.587895,-0.406313,-0.410078,0.520996,0.385752,-0.305242,0.305097,0.909933,...,2021-12,202112,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327
4091704,93434,2021-12-31,-0.624095,-0.795251,-0.799884,-0.613380,-0.358529,0.609036,0.220533,-0.931364,...,2021-12,202112,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.015640,0.001327


In [14]:
db.close()

In [15]:
df_merged.to_csv('/Users/ywo/Downloads/Industry Project/final data.csv', index=False)